# Hybrid Recommender System Projesi

ID'si verilen kullanıcı için item-based ve user-based recommender yöntemlerini kullanarak tahmin yapınız.
5 öneri user-based modelden, 5 öneri de item-based modelden ele alınız ve nihai olarak 10 öneriyi 2 modelden yapınız.

## Görevler

1. **GÖREV 1:** Verinin Hazırlanması
2. **GÖREV 2:** Öneri Yapılacak Kullanıcının İzlediği Filmlerin Belirlenmesi
3. **GÖREV 3:** Aynı Filmleri İzleyen Diğer Kullanıcıların Verisine ve Id'lerine Erişmek
4. **GÖREV 4:** Öneri Yapılacak Kullanıcı ile En Benzer Kullanıcıların Belirlenmesi
5. **GÖREV 5:** Weighted Average Recommendation Score'un Hesaplanması ve İlk 5 Filmin Tutulması
6. **GÖREV 6:** Item-Based Recommendation

In [ ]:
# ============================================================
# KÜTÜPHANE İÇE AKTARMA VE GÖRÜNTÜ AYARLARI
# ============================================================
# pandas (pd): Veri okuma, birleştirme, filtreleme ve tablo işlemleri için kullanılır.
# numpy (np): Matematiksel işlemler ve NaN (eksik değer) kontrolü için kullanılır.
import pandas as pd
import numpy as np

# Aşağıdaki ayarlar, geniş tabloları ekranda kesilmeden görmeyi sağlar.
pd.set_option('display.max_columns', None)  # Tüm sütunları göster
pd.set_option('display.width', 500)       # Satır genişliğini artır


---
## GÖREV 1: Verinin Hazırlanması

### Adım 1: Veri Setlerini Okuma

Movie ve Rating veri setlerini okutunuz.

- **movie:** `movieId`, film adı ve filmin tür bilgilerini içeren veri seti
- **rating:** `UserID`, film adı, filme verilen oy ve zaman bilgisini içeren veri seti

In [ ]:
# ============================================================
# GÖREV 1 - ADIM 1: MOVIE VERİ SETİNİ OKUMA
# ============================================================
# movie.csv dosyası her filmin kimlik bilgilerini içerir:
#   - movieId : Filmin benzersiz numarası
#   - title   : Filmin adı
#   - genres  : Filmin türleri (Aksiyon, Komedi vb.)
# read_csv() fonksiyonu CSV dosyasını okuyup pandas DataFrame'e çevirir.
movie = pd.read_csv("datasets/movie.csv")

# head() fonksiyonu varsayılan olarak ilk 5 satırı gösterir.
# Veri setinin yapısını ve sütun isimlerini anlamak için kullanılır.
movie.head()


In [ ]:
# ============================================================
# GÖREV 1 - ADIM 1: RATING VERİ SETİNİ OKUMA
# ============================================================
# rating.csv dosyası kullanıcıların filmlere verdiği puanları içerir:
#   - userId    : Kullanıcının benzersiz numarası
#   - movieId   : Puanlanan filmin numarası (movie tablosuyla eşleşir)
#   - rating    : Kullanıcının verdiği puan (0.5 ile 5 arası)
#   - timestamp : Puanın verildiği tarih/saat
rating = pd.read_csv("datasets/rating.csv")
rating.head()


In [ ]:
# movie veri setinin boyutunu kontrol ediyoruz.
# shape çıktısı: (satır_sayısı, sütun_sayısı)
# Örnek: (27278, 3) → 27278 film, 3 sütun (movieId, title, genres)
movie.shape


In [ ]:
# rating veri setinin boyutunu kontrol ediyoruz.
# Bu veri seti çok büyüktür çünkü her kullanıcı-film puanı ayrı bir satırdır.
# Örnek: (20000263, 4) → yaklaşık 20 milyon puan kaydı
rating.shape


### Adım 2: Rating Veri Setine Film Bilgilerini Ekleme

Rating veri setine filmlerin isimlerini ve türünü movie film setini kullanarak ekleyiniz.
Rating'deki kullanıcıların oy kullandıkları filmlerin sadece id'si var.
Id'lere ait film isimlerini ve türünü movie veri setinden ekliyoruz.

In [ ]:
# ============================================================
# GÖREV 1 - ADIM 2: VERİ SETLERİNİ BİRLEŞTİRME (MERGE)
# ============================================================
# rating tablosunda sadece movieId vardır; film adını bilmek için movie tablosuyla birleştiririz.
# merge(): İki DataFrame'i ortak bir sütun üzerinden birleştirir.
#   - how='left' : rating tablosundaki TÜM satırlar korunur.
#   - on='movieId': Birleştirme anahtarı movieId sütunudur.
# Sonuç: Her puan satırına film adı (title) ve türü (genres) eklenmiş olur.
df = movie.merge(rating, how="left", on="movieId")
df.head()


In [ ]:
# Birleştirme sonrası veri setinin boyutunu kontrol ediyoruz.
# Satır sayısı rating ile aynı olmalıdır; sütun sayısı movie bilgileriyle artmıştır.
df.shape


### Adım 3: Nadir Filmleri Çıkarma

Her bir film için toplam kaç kişinin oy kullandığını hesaplayınız.
Toplam oy kullanılma sayısı 1000'ün altında olan filmleri veri setinden çıkarınız.

In [ ]:
# ============================================================
# GÖREV 1 - ADIM 3: NADİR FİLMLERİ TESPİT ETME
# ============================================================
# Öneri sistemlerinde çok az kişinin puanladığı filmler güvenilir sonuç vermez.
# Bu yüzden her filmin kaç kez puanlandığını sayıyoruz.
#
# value_counts(): Bir sütundaki her değerin kaç kez geçtiğini sayar.
# DataFrame'e çevirince: index=film adı, count=oy sayısı olur.
comment_counts = pd.DataFrame(df["title"].value_counts())
comment_counts.columns = ["count"]  # Sütun adını açıkça belirliyoruz
comment_counts


In [ ]:
# ============================================================
# GÖREV 1 - ADIM 3 (devam): NADİR FİLMLERİ ÇIKARMA
# ============================================================
# 1000'den az oy alan filmler 'nadir' (rare) kabul edilir ve analizden çıkarılır.
# Bu eşik değer, modelin daha güvenilir ve yaygın filmlere odaklanmasını sağlar.
#
# comment_counts["count"] <= 1000 : Oy sayısı 1000 ve altı olan filmleri seçer
# .index : Bu filmlerin isimlerini (title) bir liste olarak döndürür
rare_movies = comment_counts[comment_counts["count"] <= 1000].index

# ~ işareti 'DEĞİL' anlamına gelir (ters filtreleme).
# isin(): Bir değerin listede olup olmadığını kontrol eder.
# Sonuç: Sadece 1000'den fazla oy alan 'yaygın' (common) filmler kalır.
common_movies = df[~df["title"].isin(rare_movies)]

# Filtreleme sonrası kaç satır kaldığını kontrol ediyoruz.
common_movies.shape


### Adım 4: Pivot Table Oluşturma

Index'te `userID`'lerin, sütunlarda film isimlerinin ve değer olarak rating'lerin bulunduğu dataframe için pivot table oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 1 - ADIM 4: PIVOT TABLE (KULLANICI-FİLM MATRİSİ) OLUŞTURMA
# ============================================================
# Öneri sistemlerinin temel yapı taşıdır: Kullanıcı-Film Puan Matrisi
#
# pivot_table() uzun formattaki veriyi geniş formata çevirir:
#   - index  = userId    → Her SATIR bir kullanıcıyı temsil eder
#   - columns= title     → Her SÜTUN bir filmi temsil eder
#   - values = rating    → Hücredeki değer, o kullanıcının o filme verdiği puandır
#
# Kullanıcı bir filmi izlememişse hücre NaN (boş) olur.
# Bu matris hem user-based hem item-based öneriler için kullanılacaktır.
user_movie_df = common_movies.pivot_table(index=["userId"], columns=["title"], values="rating")
user_movie_df.head()


### Adım 5: Fonksiyonlaştırma

Yukarıda yapılan tüm işlemleri fonksiyonlaştıralım.

In [ ]:
# ============================================================
# GÖREV 1 - ADIM 5: TÜM ADIMLARI FONKSİYONLAŞTIRMA
# ============================================================
# Yukarıdaki adımları tek bir fonksiyonda topluyoruz.
# Böylece ileride user_movie_df'yi tek satırda yeniden oluşturabiliriz.
def create_user_movie_df():
    """
    Kullanıcı-film puan matrisini oluşturan fonksiyon.

    İşlem sırası:
    1. movie ve rating veri setlerini oku
    2. movieId üzerinden birleştir
    3. 1000'den az oy alan nadir filmleri çıkar
    4. Kalan veriden pivot table oluştur

    Returns:
        user_movie_df: Satırlarda kullanıcı, sütunlarda film, değerlerde puan olan DataFrame
    """
    movie = pd.read_csv("datasets/movie.csv")
    rating = pd.read_csv("datasets/rating.csv")
    df = movie.merge(rating, how="left", on="movieId")
    comment_counts = pd.DataFrame(df["title"].value_counts())
    comment_counts.columns = ["count"]
    rare_movies = comment_counts[comment_counts["count"] <= 1000].index
    common_movies = df[~df["title"].isin(rare_movies)]
    user_movie_df = common_movies.pivot_table(index=["userId"], columns=["title"], values="rating")
    return user_movie_df

# Fonksiyonu çalıştırıp matrisi oluşturuyoruz.
# NOT: Büyük veri olduğu için bu adım 30-60 saniye sürebilir.
user_movie_df = create_user_movie_df()
user_movie_df.shape


---
## GÖREV 2: Öneri Yapılacak Kullanıcının İzlediği Filmlerin Belirlenmesi

### Adım 1: Rastgele Kullanıcı Seçimi

Rastgele bir kullanıcı id'si seçiniz.

In [ ]:
# ============================================================
# GÖREV 2 - ADIM 1: RASTGELE KULLANICI SEÇİMİ
# ============================================================
# User-based öneri yapacağımız hedef kullanıcıyı belirliyoruz.
#
# user_movie_df.index → Pivot tablodaki tüm kullanıcı ID'leri
# pd.Series()         → Bu ID'leri pandas Series'e çeviriyoruz (sample için gerekli)
# .sample(n=1)       → Listeden rastgele 1 kullanıcı seçer
# random_state=45     → Her çalıştırmada AYNI kullanıcıyı seçer (tekrarlanabilirlik)
# .iloc[0]            → Seçilen değeri Series'ten çıkarıp düz sayı olarak alır
random_user = pd.Series(user_movie_df.index).sample(n=1, random_state=45).iloc[0]
random_user


### Adım 2: Kullanıcı Gözlem Birimlerini Filtreleme

Seçilen kullanıcıya ait gözlem birimlerinden oluşan `random_user_df` adında yeni bir dataframe oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 2 - ADIM 2: SEÇİLEN KULLANICININ VERİSİNİ FİLTRELEME
# ============================================================
# user_movie_df'ten sadece random_user'a ait satırı çekiyoruz.
#
# user_movie_df.index == random_user → True/False maskesi oluşturur
# Bu maskeyle filtreleme yapınca sadece hedef kullanıcının satırı kalır.
# Sonuç: 1 satır x binlerce sütun (her sütun bir film, değer=puan veya NaN)
random_user_df = user_movie_df[user_movie_df.index == random_user]
random_user_df.head()


In [ ]:
# Seçilen kullanıcının toplam kaç filme puan verdiğini kontrol ediyoruz.
# notna()  → NaN olmayan (yani puan verilmiş) hücreleri True yapar
# sum()    → True değerlerini sayar
# axis=1   → Satır bazında say (her kullanıcı için ayrı ayrı)
random_user_df.notna().sum(axis=1).values[0]


### Adım 3: İzlenen Filmleri Listeleme

Seçilen kullanıcının oy kullandığı filmleri `movies_watched` adında bir listeye atayınız.

In [ ]:
# ============================================================
# GÖREV 2 - ADIM 3: İZLENEN FİLMLERİ LİSTELEME
# ============================================================
# Kullanıcının puan verdiği filmlerin isimlerini bir listeye alıyoruz.
#
# notna().any() → Sütunda en az bir dolu hücre varsa True (bu film izlenmiş demektir)
# .columns[...] → True olan sütunların isimlerini (film adlarını) alır
# .tolist()     → pandas Index'i Python listesine çevirir
movies_watched = random_user_df.columns[random_user_df.notna().any()].tolist()
movies_watched[:5]  # Listenin ilk 5 filmini gösteriyoruz


---
## GÖREV 3: Aynı Filmleri İzleyen Diğer Kullanıcıların Verisine ve Id'lerine Erişmek

### Adım 1: İzlenen Filmlere Ait Sütunları Seçme

Seçilen kullanıcının izlediği filmlere ait sütunları `user_movie_df`'ten seçiniz ve `movies_watched_df` adında yeni bir dataframe oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 3 - ADIM 1: İZLENEN FİLMLERE AİT SÜTUNLARI SEÇME
# ============================================================
# user_movie_df'ten SADECE random_user'ın izlediği filmlerin sütunlarını alıyoruz.
#
# Satırlar: Tüm kullanıcılar (herkesin bu filmlere verdiği puanlar)
# Sütunlar: Sadece hedef kullanıcının izlediği filmler
# Amaç: 'Bu filmleri kimler izlemiş?' sorusuna cevap aramak
movies_watched_df = user_movie_df[movies_watched]
movies_watched_df.shape


### Adım 2: Kullanıcı Film Sayısı Hesaplama

Her bir kullanıcının seçili user'in izlediği filmlerin kaçını izlediği bilgisini taşıyan `user_movie_count` adında yeni bir dataframe oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 3 - ADIM 2: KULLANICI BAŞINA ORTAK FİLM SAYISI
# ============================================================
# Her kullanıcının, random_user ile kaç ortak film izlediğini hesaplıyoruz.
#
# notnull().sum(axis=1) → Her satırda (kullanıcıda) kaç dolu hücre var?
# reset_index()         → Index'teki userId'yi normal sütuna çevirir
# columns = [...]       → Sütun isimlerini anlamlı hale getirir
user_movie_count = movies_watched_df.notnull().sum(axis=1).reset_index()
user_movie_count.columns = ["userId", "movie_count"]
user_movie_count.sort_values("movie_count", ascending=False).head()


### Adım 3: Benzer Kullanıcıları Belirleme

Seçilen kullanıcının oy verdiği filmlerin yüzde 60 ve üstünü izleyenleri benzer kullanıcılar olarak görüyoruz.
Bu kullanıcıların id'lerinden `users_same_movies` adında bir liste oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 3 - ADIM 3: BENZER KULLANICILARI BELİRLEME
# ============================================================
# random_user'ın izlediği filmlerin en az %60'ını izleyenleri 'benzer kullanıcı' sayıyoruz.
#
# Örnek: Kullanıcı 100 film izlemişse → eşik = 100 * 60/100 = 60 film
# 60'tan fazla ortak filmi olan kullanıcılar benzer kabul edilir.
# Bu eşik ne kadar yüksekse, o kadar 'benzer' ama daha az kullanıcı bulunur.
perc = len(movies_watched) * 60 / 100
users_same_movies = user_movie_count[user_movie_count["movie_count"] > perc]["userId"]
len(users_same_movies)  # Kaç benzer kullanıcı bulundu?


---
## GÖREV 4: Öneri Yapılacak Kullanıcı ile En Benzer Kullanıcıların Belirlenmesi

### Adım 1: Benzer Kullanıcıları Filtreleme

`users_same_movies` listesi içerisindeki seçili user ile benzerlik gösteren kullanıcıların id'lerinin bulunacağı şekilde `movies_watched_df` dataframe'ini filtreleyiniz.

In [ ]:
# ============================================================
# GÖREV 4 - ADIM 1: BENZER KULLANICILARI FİLTRELEME
# ============================================================
# movies_watched_df'ten sadece benzer kullanıcıların satırlarını tutuyoruz.
#
# .index.isin(users_same_movies) → Satırın userId'si benzer kullanıcı listesinde mi?
# Bu filtreleme ile random_user hariç, sadece benzer zevke sahip kullanıcılar kalır.
movies_watched_df = movies_watched_df[movies_watched_df.index.isin(users_same_movies)]
movies_watched_df.shape


### Adım 2: Korelasyon DataFrame'i Oluşturma

Kullanıcıların birbirleri ile olan korelasyonlarının bulunacağı yeni bir `corr_df` dataframe'i oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 4 - ADIM 2: KULLANICILAR ARASI KORELASYON HESAPLAMA
# ============================================================
# Korelasyon: İki kullanıcının film zevklerinin ne kadar benzer olduğunu ölçer.
# +1'e yakın = çok benzer zevk, 0 = ilişkisiz, -1 = tam tersi zevk
#
# ADIM A: random_user'ı da ekleyerek final_df oluşturuyoruz (korelasyon için gerekli)
final_df = pd.concat([movies_watched_df, random_user_df[movies_watched]])

# ADIM B: .T (transpose/devrik) → Satır ve sütunları yer değiştirir
#         Böylece kullanıcılar satır, filmler sütun olur → .corr() kullanıcılar arası korelasyonu hesaplar
corr_matrix = final_df.T.corr()

# ADIM C: Korelasyon matrisini uzun formata (çift bazlı) çeviriyoruz
# Her kullanıcı çifti için korelasyon değerini ayrı bir satır olarak kaydediyoruz
corr_pairs = []
index_array = corr_matrix.index.to_numpy()
for i in range(len(index_array)):
    for j in range(i + 1, len(index_array)):  # i+1: Her çifti sadece bir kez al
        corr_value = corr_matrix.iloc[i, j]
        if not np.isnan(corr_value):  # NaN = ortak puanlanmış film yok, atla
            corr_pairs.append((index_array[i], index_array[j], corr_value))

corr_df = pd.DataFrame(corr_pairs, columns=["user_id_1", "user_id_2", "correlation"])
corr_df = corr_df.sort_values("correlation", ascending=False).drop_duplicates()
corr_df.head()


### Adım 3: Yüksek Korelasyonlu Kullanıcıları Filtreleme

Seçili kullanıcı ile yüksek korelasyona sahip (0.65'in üzerinde olan) kullanıcıları filtreleyerek `top_users` adında yeni bir dataframe oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 4 - ADIM 3: YÜKSEK KORELASYONLU KULLANICILARI SEÇME
# ============================================================
# random_user ile korelasyonu 0.65 ve üzeri olan kullanıcıları 'en benzer' kabul ediyoruz.
# 0.65 eşiği: Çok benzer zevke sahip kullanıcıları ayıklar, gürültüyü azaltır.
#
# Filtreleme: user_id_1 == random_user VE correlation >= 0.65
# Sadece user_id_2 (benzer kullanıcı) ve correlation sütunlarını alıyoruz
top_users = corr_df[
    (corr_df["user_id_1"] == random_user) & (corr_df["correlation"] >= 0.65)
][["user_id_2", "correlation"]].reset_index(drop=True)

top_users = top_users.sort_values(by="correlation", ascending=False)
top_users.rename(columns={"user_id_2": "userId"}, inplace=True)  # Sütun adını standartlaştır
top_users.head()


### Adım 4: Rating Verisi ile Merge

`top_users` dataframe'ine rating veri seti ile merge ediniz.

In [ ]:
# ============================================================
# GÖREV 4 - ADIM 4: BENZER KULLANICILARIN PUANLARINI GETİRME (MERGE)
# ============================================================
# top_users'ta sadece userId ve correlation var; bu kullanıcıların hangi filmlere
# kaç puan verdiğini öğrenmek için rating tablosuyla birleştiriyoruz.
#
# merge(how='inner') → Sadece her iki tabloda da bulunan userId'ler kalır
# Son satır: random_user'ı listeden çıkarıyoruz (kendine öneri yapmamak için)
rating = pd.read_csv("datasets/rating.csv")
top_users_ratings = top_users.merge(rating[["userId", "movieId", "rating"]], how="inner")
top_users_ratings = top_users_ratings[top_users_ratings["userId"] != random_user]
top_users_ratings.head()


---
## GÖREV 5: Weighted Average Recommendation Score'un Hesaplanması ve İlk 5 Filmin Tutulması

### Adım 1: Weighted Rating Oluşturma

Her bir kullanıcının corr ve rating değerlerinin çarpımından oluşan `weighted_rating` adında yeni bir değişken oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 5 - ADIM 1: AĞIRLIKLI PUAN (WEIGHTED RATING) HESAPLAMA
# ============================================================
# Her benzer kullanıcının puanını, random_user ile olan korelasyonuyla çarpıyoruz.
#
# Mantık: random_user'a ÇOK benzeyen birinin 5 puanı, az benzeyenin 5 puanından daha değerlidir.
# Formül: weighted_rating = correlation × rating
#
# Örnek: Korelasyon=0.90, Rating=5 → weighted_rating = 4.50 (yüksek güven)
# Örnek: Korelasyon=0.70, Rating=5 → weighted_rating = 3.50 (daha düşük güven)
top_users_ratings["weighted_rating"] = top_users_ratings["correlation"] * top_users_ratings["rating"]
top_users_ratings[["userId", "movieId", "rating", "correlation", "weighted_rating"]].head()


### Adım 2: Recommendation DataFrame Oluşturma

Film id'si ve her bir filme ait tüm kullanıcıların weighted rating'lerinin ortalama değerini içeren `recommendation_df` adında yeni bir dataframe oluşturunuz.

In [ ]:
# ============================================================
# GÖREV 5 - ADIM 2: FİLM BAZINDA ÖNERİ SKORU HESAPLAMA
# ============================================================
# Her film için, onu puanlayan tüm benzer kullanıcıların weighted_rating ortalamasını alıyoruz.
#
# groupby('movieId') → Aynı filmi puanlayan satırları gruplar
# agg({'weighted_rating': 'mean'}) → Her grup için ortalama hesaplar
# Sonuç: Her filmin tek bir öneri skoru (weighted_rating ortalaması) olur
recommendation_df = top_users_ratings.groupby("movieId").agg({"weighted_rating": "mean"}).reset_index()
recommendation_df.sort_values("weighted_rating", ascending=False).head()


### Adım 3: İlk 5 Filmi Seçme

`recommendation_df` içerisinde weighted rating'i 3.5'ten büyük olan filmleri seçiniz ve weighted rating'e göre sıralayınız.
İlk 5 gözlemi `movies_to_be_recommend` olarak kaydediniz.

In [ ]:
# ============================================================
# GÖREV 5 - ADIM 3: USER-BASED İLK 5 FİLM ÖNERİSİ
# ============================================================
# Kalite filtresi: weighted_rating > 3.5 (düşük skorlu filmleri eliyoruz)
# Sıralama: En yüksek skordan en düşüğe
# .head(5): İlk 5 filmi al → User-based modelden 5 öneri
movies_to_be_recommend = (
    recommendation_df[recommendation_df["weighted_rating"] > 3.5]
    .sort_values("weighted_rating", ascending=False)
    .head(5)
)
movies_to_be_recommend


### Adım 4: Tavsiye Edilen Filmlerin İsimlerini Getirme

Tavsiye edilen 5 filmin isimlerini getiriniz.

In [ ]:
# ============================================================
# GÖREV 5 - ADIM 4: ÖNERİLEN FİLMLERİN İSİMLERİNİ GETİRME
# ============================================================
# movies_to_be_recommend'te sadece movieId ve skor var.
# merge() ile movie tablosundan film isimlerini (title) ekliyoruz.
movie = pd.read_csv("datasets/movie.csv")
user_based_recommendations = movies_to_be_recommend.merge(movie[["movieId", "title"]])
user_based_recommendations


---
## GÖREV 6: Item-Based Recommendation

Kullanıcının en son izlediği ve en yüksek puan verdiği filmin adına göre item-based öneri yapınız.

In [ ]:
# ============================================================
# GÖREV 6 BAŞLANGICI
# ============================================================
# Item-based öneri için aynı random_user kullanılmaya devam edilir.
# User-based: 'Benzer kullanıcılar ne izledi?' sorusuna cevap verir.
# Item-based:  'Bu filme benzeyen filmler hangileri?' sorusuna cevap verir.
print(f"Item-based öneri için hedef kullanıcı: {int(random_user)}")


### Adım 1: Veri Setlerini Okuma

Movie ve rating veri setlerini okutunuz.

In [ ]:
# ============================================================
# GÖREV 6 - ADIM 1: VERİ SETLERİNİ YENİDEN OKUMA
# ============================================================
# Item-based bölümünde timestamp (tarih) bilgisine ihtiyacımız var.
# user_movie_df'te tarih bilgisi yok; bu yüzden rating tablosunu tekrar okuyoruz.
movie = pd.read_csv("datasets/movie.csv")
rating = pd.read_csv("datasets/rating.csv")
rating.head()


### Adım 2: En Güncel Yüksek Puanlı Filmi Bulma

Öneri yapılacak kullanıcının 5 puan verdiği filmlerden puanı en güncel olan filmin id'sini alınız.

In [ ]:
# ============================================================
# GÖREV 6 - ADIM 2: EN GÜNCEL 5 YILDIZLI FİLMİ BULMA
# ============================================================
# Item-based önerinin referans noktası: Kullanıcının en son beğendiği (5 puan) film.
#
# ADIM A: random_user'ın 5 puan verdiği tüm filmleri filtrele
user_5_star = rating[(rating["userId"] == random_user) & (rating["rating"] == 5)].copy()

# ADIM B: timestamp sütununu tarih formatına çevir (sıralama için gerekli)
user_5_star["timestamp"] = pd.to_datetime(user_5_star["timestamp"])

# ADIM C: Tarihe göre azalan sırala → en güncel film en üstte
# .iloc[0] ile ilk satırı al → en son 5 yıldız verilen filmin movieId'si
last_movie_id = user_5_star.sort_values("timestamp", ascending=False).iloc[0]["movieId"]
last_movie_id


### Adım 3: user_movie_df Filtreleme

User based recommendation bölümünde oluşturulan `user_movie_df` dataframe'ini seçilen film id'sine göre filtreleyiniz.

In [ ]:
# ============================================================
# GÖREV 6 - ADIM 3: SEÇİLEN FİLMİN PUAN VEKTÖRÜNÜ ALMA
# ============================================================
# user_movie_df sütunları film adıdır, movieId değil.
# Bu yüzden önce movieId → film adı (title) dönüşümü yapıyoruz.
#
# selected_movie: Tüm kullanıcıların bu filme verdiği puanlardan oluşan bir sütun (Series)
# Bu vektör, item-based korelasyon hesabının temel girdisidir.
movie_name = movie[movie["movieId"] == last_movie_id]["title"].values[0]
selected_movie = user_movie_df[movie_name]
selected_movie.head()


### Adım 4: Film Korelasyonlarını Hesaplama

Filtrelenen dataframe'i kullanarak seçili filmle diğer filmlerin korelasyonunu bulunuz ve sıralayınız.

In [ ]:
# ============================================================
# GÖREV 6 - ADIM 4: FİLMLER ARASI KORELASYON HESAPLAMA
# ============================================================
# Item-based filtreleme: 'Bu filme benzeyen filmler hangileri?'
#
# corrwith(selected_movie): selected_movie sütunuyla diğer TÜM film sütunlarının
# korelasyonunu hesaplar. Yüksek korelasyon = benzer izleyici kitlesi = benzer film.
# sort_values(ascending=False): En benzer filmler en üstte
correlated_movies = user_movie_df.corrwith(selected_movie).sort_values(ascending=False)
correlated_movies.head(10)


### Adım 5: İlk 5 Film Önerisi

Seçili film'in kendisi haricinde ilk 5 film'i öneri olarak veriniz.

In [ ]:
# ============================================================
# GÖREV 6 - ADIM 5: ITEM-BASED İLK 5 FİLM ÖNERİSİ
# ============================================================
# .iloc[0] → Referans filmin kendisi (korelasyon = 1.0), öneri listesine dahil edilmez
# .iloc[1:6] → Kendisi hariç en yüksek korelasyona sahip 5 film
# Sonuç: Item-based modelden 5 öneri
item_based_recommendations = correlated_movies.iloc[1:6]
item_based_recommendations


---
## Nihai Hybrid Öneri (10 Film)

User-based modelden 5, item-based modelden 5 film birleştirilerek toplam 10 öneri üretilir.

In [ ]:
# ============================================================
# NİHAİ HYBRID ÖNERİ: USER-BASED + ITEM-BASED (TOPLAM 10 FİLM)
# ============================================================
# Hybrid (hibrit) sistem: İki farklı öneri yaklaşımının sonuçlarını birleştirir.
#   - User-based (5 film): Benzer zevkteki kullanıcıların beğendiği filmler
#   - Item-based  (5 film): Referans filme benzeyen filmler
#
# Bu yaklaşım hem kişiselleştirme hem de içerik benzerliğini bir arada sunar.
print("=== USER-BASED (5 Film) ===")
print(user_based_recommendations[["title", "weighted_rating"]].to_string(index=False))

print("\n=== ITEM-BASED (5 Film) ===")
print(item_based_recommendations.to_string())

print(f"\nReferans film (item-based): {movie_name}")
print(f"Hedef kullanıcı: {int(random_user)}")
